In [ ]:
import numpy as np
import heapq

# ── Configuration ────────────────────────────────────────────────────────────
GRID   = 70        # grid side length (km)
OBS    = 0.2       # obstacle density  (0.0 – 1.0)
ORIGIN = (0, 0)
TARGET = (GRID - 1, GRID - 1)

# Move table: (row-delta, col-delta, cost)
MOVES = [
    ( 0,  1, 1.00), ( 0, -1, 1.00),
    ( 1,  0, 1.00), (-1,  0, 1.00),
    ( 1,  1, 1.41), ( 1, -1, 1.41),
    (-1,  1, 1.41), (-1, -1, 1.41),
]

# ── Map generation ───────────────────────────────────────────────────────────
def make_terrain(rows=GRID, density=OBS, seed=None):
    rng = np.random.default_rng(seed)
    terrain = rng.choice([0, 1], size=(rows, rows), p=[1 - density, density])
    terrain[ORIGIN] = 0
    terrain[TARGET] = 0
    return terrain

# ── Dijkstra ─────────────────────────────────────────────────────────────────
def find_route(terrain, src, dst):
    rows = terrain.shape[0]
    best = {}                          # node -> best cost seen
    came_from = {}                     # for path reconstruction
    pq = [(0.0, src)]
    best[src] = 0.0

    while pq:
        cost, node = heapq.heappop(pq)
        if cost > best.get(node, float('inf')):
            continue
        if node == dst:
            break
        r, c = node
        for dr, dc, w in MOVES:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < rows and terrain[nr, nc] == 0:
                new_cost = cost + w
                if new_cost < best.get((nr, nc), float('inf')):
                    best[(nr, nc)] = new_cost
                    came_from[(nr, nc)] = node
                    heapq.heappush(pq, (new_cost, (nr, nc)))

    if dst not in came_from and dst != src:
        return None, None, best

    # Reconstruct path
    path = []
    cur = dst
    while cur != src:
        path.append(cur)
        cur = came_from[cur]
    path.append(src)
    path.reverse()
    return best.get(dst), path, best

# ── Run static scenario ───────────────────────────────────────────────────────
terrain_map = make_terrain()
static_cost, static_path, visited_nodes = find_route(terrain_map, ORIGIN, TARGET)

In [ ]:
import matplotlib.pyplot as plt

# ── MoE calculations ─────────────────────────────────────────────────────────
straight_line = np.sqrt(2) * (GRID - 1)
pct_explored  = len(visited_nodes) / GRID**2 * 100

if static_cost:
    efficiency = straight_line / static_cost * 100
else:
    efficiency = 0
    static_cost = 0

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(terrain_map, cmap='binary', origin='lower')

if visited_nodes:
    vr, vc = zip(*visited_nodes.keys())
    ax.scatter(vc, vr, color='cyan', s=1, alpha=0.2, label='Explored')

if static_path:
    pr, pc = zip(*static_path)
    ax.plot(pc, pr, color='red', linewidth=3, label='Optimal Path')
    ax.set_title(f'MISSION SUCCESS  (density={OBS*100:.0f}%)', fontsize=14)
else:
    ax.set_title(f'MISSION FAILURE — Path Blocked  (density={OBS*100:.0f}%)',
                 fontsize=14, color='red')

ax.scatter(ORIGIN[1], ORIGIN[0], color='green', s=150, zorder=5, label='Start')
ax.scatter(TARGET[1], TARGET[0], color='blue',  s=200, zorder=5,
           marker='*', label='Goal')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

# ── Print MoE ─────────────────────────────────────────────────────────────────
print(f"\n{' MEASURES OF EFFECTIVENESS ':=^42}")
print(f"  Path Distance :  {static_cost:.2f} km")
print(f"  Path Efficiency: {efficiency:.2f}%  (vs straight line)")
print(f"  Search Effort :  {pct_explored:.2f}%  of map explored")
print('=' * 42)

if not static_path:
    print('All reachable cells exhausted — no route to target.')

In [ ]:
import numpy as np

NUM_PATROLS  = 40
STEP_LIMIT   = GRID * 4   # thrashing guard

def simulate_dynamic(base_map, start, goal, n_patrols=NUM_PATROLS):
    pos       = start
    trail     = [start]
    distance  = 0.0

    # Scatter patrols randomly, away from start/goal
    rng     = np.random.default_rng()
    patrols = []
    while len(patrols) < n_patrols:
        r, c = rng.integers(0, GRID, size=2)
        if (r, c) not in (start, goal):
            patrols.append([r, c])

    step = 0
    while pos != goal:
        # Move patrols (+1 on column axis, wrap at boundary)
        live_map = base_map.copy()
        for p in patrols:
            p[1] = (p[1] + 1) % GRID
            live_map[p[0], p[1]] = 1

        # Replan from current position
        cost_to_go, route, _ = find_route(live_map, pos, goal)

        if not route:
            print(f'MISSION FAILED: UGV cornered at step {len(trail)}.')
            return distance, trail, False

        # Advance one step
        nxt        = route[1]
        distance  += np.hypot(nxt[0] - pos[0], nxt[1] - pos[1])
        pos        = nxt
        trail.append(pos)
        step      += 1

        if step >= STEP_LIMIT:
            print('MISSION FAILED: thrashing limit reached.')
            return distance, trail, False

    print('MISSION SUCCESS: goal reached!')
    return distance, trail, True

dyn_dist, dyn_trail, dyn_ok = simulate_dynamic(terrain_map, ORIGIN, TARGET)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(terrain_map, cmap='binary', origin='lower')

if dyn_trail:
    tr, tc = zip(*dyn_trail)
    ax.plot(tc, tr, color='orange', linewidth=2.5,
            linestyle='--', label='Dynamic Path')

ax.scatter(ORIGIN[1], ORIGIN[0], color='green', s=150, zorder=5, label='Start')
ax.scatter(TARGET[1], TARGET[0], color='blue',  s=200, zorder=5,
           marker='*', label='Goal')

status = 'SUCCESS' if dyn_ok else 'FAILED (trapped / thrashing)'
ax.set_title(f'Dynamic UGV Navigation — {status}', fontsize=14)
ax.legend(loc='upper left')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n{' DYNAMIC MoE ':=^42}")
print(f"  Steps taken    : {len(dyn_trail)}")
print(f"  Total distance : {dyn_dist:.2f} km")
if dyn_ok and static_cost:
    print(f"  Evasive penalty: +{dyn_dist - static_cost:.2f} km")
print('=' * 42)